[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sharukh01621/first-line-to-first-paper/blob/main/colab_drive/part_06_deep_learning/ch31_sequences_rnns_time_series/03_coding_exam.ipynb)

> **From First Line to First Paper** — *Mastering Python, Statistical Data Science, and Applied Deep Learning*  
> **Author:** Mezanur Rahman Sharukh | **Mode:** `INTERACTIVE STUDENT NOTEBOOK`


# Chapter 31 Coding Exam — Epidemiological Sequence Modeling & Forecasting (100 Points)

**Book Reference:** Chapter 31, Section 31.7 (Coding Exam)  
**Suggested Time:** 45–55 minutes (Open Book, Closed Solution)

| Task | Sequence Forecasting Pipeline Component | Points |
| :--- | :--- | :---: |
| **Task 31.1** | `temporal_district_window_split(epi_df, district, seq_len=8, cutoff_year=2025)` (Leak-free temporal windows) | 30 |
| **Task 31.2** | `train_epi_gru_forecaster(X_train, y_train, X_test, y_test, ...)` (Clipped PyTorch GRU training loop) | 35 |
| **Task 31.3** | `compare_gru_lstm_persistence_epi(epi_df, district="Dhaka", seq_len=8, seed=42)` (3-model benchmark) | 35 |
| **TOTAL** | | **100** |

In [ ]:
# @title Step 0: Click Play (▶) to Activate Chapter 31 Exam Grader
!pip install -q "git+https://github.com/sharukh01621/first-line-to-first-paper.git#subdirectory=sharukh_tutor"
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
import sharukh_tutor as tutor

torch.set_num_threads(1)
tutor.activate(chapter="ch31", notebook="coding_exam")


---
## Exam Task 31.1 (30 Points): `temporal_district_window_split(epi_df, district, seq_len=8, cutoff_year=2025)`

Build a leakage-free chronological sliding-window split for a single district in `epi_surveillance` (without mutating `epi_df`):
1. Filter `epi_df[epi_df["district"] == district]` and sort chronologically by `["year", "epi_week"]`.
2. Extract numeric features in exact column order `["dengue_cases", "rainfall_mm", "mean_temp_c"]` (index `0` is the target `dengue_cases`).
3. Fit `StandardScaler()` **strictly on rows where `year < cutoff_year`**, then transform the full sorted feature matrix `scaled = scaler.transform(raw)`.
4. Build standardized sliding windows (both inputs and targets on the training-fitted `StandardScaler` scale):
   - **Train windows** strictly inside `year < cutoff_year`: for `t` in `range(seq_len, n_train)`, `X_train` window is `scaled[t - seq_len : t]` and `y_train` is `scaled[t, 0]`.
   - **Test windows** for `year >= cutoff_year`: for `t` in `range(n_train, len(sub))`, `X_test` window is `scaled[t - seq_len : t]` (strictly preceding history!) and `y_test` is `scaled[t, 0]`.
5. Return `{"X_train": X_train, "y_train": y_train, "X_test": X_test, "y_test": y_test, "scaler": scaler}`.

In [ ]:
def temporal_district_window_split(
    epi_df, district, seq_len=8, cutoff_year=2025
):
    """Leak-free temporal train/test sliding windows (standardized)."""
    # YOUR CODE HERE
    raise NotImplementedError("Replace with your solution")


---
## Exam Task 31.2 (35 Points): `train_epi_gru_forecaster(X_train, y_train, X_test, y_test, hidden_size=16, epochs=40, lr=0.02, seed=42)`

Train a PyTorch GRU forecaster cleanly with gradient clipping:
1. Seed PyTorch via `torch.manual_seed(seed)` and construct a GRU model (`nn.GRU(input_size, hidden_size, batch_first=True)` + `nn.Linear(hidden_size, 1)` reading `out[:, -1, :]`).
2. Train for `epochs` epochs with `nn.MSELoss()`, `torch.optim.Adam(model.parameters(), lr=lr)`, and `nn.utils.clip_grad_norm_(model.parameters(), 1.0)` before each `optimizer.step()`.
3. Switch to `model.eval()` and `with torch.no_grad():` to compute 1D float NumPy predictions `preds` on `X_test`.
4. Return `{"test_rmse": round4, "test_mae": round4, "final_train_loss": round5, "preds": preds}`.

In [ ]:
class _ExamGRU(nn.Module):
    def __init__(self, input_size, hidden_size=16):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, batch_first=True)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        return self.head(out[:, -1, :])

def train_epi_gru_forecaster(
    X_train, y_train, X_test, y_test,
    hidden_size=16, epochs=40, lr=0.02, seed=42,
):
    """Train GRU forecaster with MSELoss, Adam & gradient clipping."""
    # YOUR CODE HERE
    raise NotImplementedError("Replace with your solution")


---
## Exam Task 31.3 (35 Points): `compare_gru_lstm_persistence_epi(epi_df, district="Dhaka", seq_len=8, seed=42)`

Run a complete leak-free epidemiological sequence benchmark (without mutating `epi_df`):
1. Call `temporal_district_window_split(epi_df, district=district, seq_len=seq_len, cutoff_year=2025)`.
2. Evaluate three models on the exact same held-out test set `(X_test, y_test)`:
   - `"Persistence"`: predicts `X_test[:, -1, 0]` (previous week's standardized dengue cases).
   - `"GRU"`: `train_epi_gru_forecaster(..., hidden_size=16, epochs=40, lr=0.02, seed=seed)`.
   - `"LSTM"`: same training setup (`hidden_size=16, epochs=40, lr=0.02, seed=seed`, `MSELoss`, `Adam`, `clip_grad_norm_(..., 1.0)`) using `nn.LSTM(..., batch_first=True)` + `nn.Linear(hidden_size, 1)`.
3. Build `leaderboard` as a `pd.DataFrame` with exact columns `["model", "test_rmse", "test_mae"]` (4 decimals), sorted ascending by `test_rmse` (`reset_index(drop=True)`).
4. Return `{"leaderboard": leaderboard, "best_model": str(leaderboard.loc[0, "model"])}`.

In [ ]:
class _ExamLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=16):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :])

def compare_gru_lstm_persistence_epi(
    epi_df, district="Dhaka", seq_len=8, seed=42
):
    """Benchmark Persistence vs GRU vs LSTM on held-out test weeks."""
    # YOUR CODE HERE
    raise NotImplementedError("Replace with your solution")


In [ ]:
exam_submissions = {
    "ch31_exam_01": temporal_district_window_split,
    "ch31_exam_02": train_epi_gru_forecaster,
    "ch31_exam_03": compare_gru_lstm_persistence_epi,
}

tutor.check_exam("ch31", exam_submissions)
